## Note — Row Format vs Columnar Format

How the *same* table is laid out on disk depends on the file format.

| EMP_ID | EMP_NAME | SALARY |
|--------|----------|--------|
| A001   | DEXTER   | 500    |
| A002   | TOM      | 600    |
| A003   | JERRY    | 1000   |

**ROW FORMAT** — all the values of one *row* stored together:

```
A001,DEXTER,500,A002,TOM,600,A003,JERRY,1000
```

**COLUMNAR FORMAT** — all the values of one *column* stored together:

```
A001,A002,A003,DEXTER,TOM,JERRY,500,600,1000
```
The columnar format lets the reader read, decompress, and process only the columns that are required for the current query.

> \*Just a simple representation, the storage logic are bit more complex than this

READING PARQUET FILES

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ParquetExample").master("local[*]").getOrCreate()

In [ ]:
# Reading Parquet Files
df_parquet = spark.read.format("parquet").load("data/parquet/sales_data.parquet")
df_parquet.printSchema()
df_parquet.show()

root
 |-- transacted_at: timestamp (nullable = true)
 |-- trx_id: integer (nullable = true)
 |-- retailer_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: integer (nullable = true)

+-------------------+----------+-----------+--------------------+-------+----------+
|      transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+-------------------+----------+-----------+--------------------+-------+----------+
|2017-12-02 19:00:00|1734117000| 1953761884|   Target ppd id: 95| 422.92| 320106707|
|2017-11-25 19:00:00|1734117016|  644879053|       Home Depot 55| 659.82|1185242216|
|2017-12-16 19:00:00|1734117666|  902350112|         Walgreen 26| 1637.1| 573392624|
|2017-12-19 19:00:00|1734118338| 2077350195|           Costco 36|2277.63| 442865762|
|2017-11-28 19:00:00|1734117444|  386167994| Best Buy ccd id: 36| 1013.1| 299170184|
|2017-12-21 19:00:00|1734117885|  562903158|     unkn ppd id: 

In [ ]:
# reading a single parquet file.
# Every part file is completely self-contained: it carries the full schema and its own rows.
# There is no separate header file.
df_parquet = spark.read.format("parquet").load("data/parquet/sales_data_multifile.parquet/part-00000-c01a2917-1539-4006-a495-b78a953d4630-c000.snappy.parquet")
df_parquet.printSchema()
df_parquet.show()

root
 |-- transacted_at: timestamp (nullable = true)
 |-- trx_id: integer (nullable = true)
 |-- retailer_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: integer (nullable = true)

+-------------------+----------+-----------+-----------+-------+----------+
|      transacted_at|    trx_id|retailer_id|description| amount|   city_id|
+-------------------+----------+-----------+-----------+-------+----------+
|2017-12-08 19:00:00|1734121607| 1898522855| unkn 11-25|2913.38|1948510326|
+-------------------+----------+-----------+-----------+-------+----------+



In [ ]:
# reading multiple parquet files.
df_parquet = spark.read.format("parquet").load("data/parquet/sales_data_multifile.parquet/*.parquet")
df_parquet.printSchema()
df_parquet.show()

root
 |-- transacted_at: timestamp (nullable = true)
 |-- trx_id: integer (nullable = true)
 |-- retailer_id: integer (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: integer (nullable = true)

+-------------------+----------+-----------+--------------------+-------+----------+
|      transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+-------------------+----------+-----------+--------------------+-------+----------+
|2017-12-21 19:00:00|1734117416| 1429095612|DineEquity ppd id...| 837.53|1465121943|
|2017-12-22 19:00:00|1734121950|  386167994| Walgreen ppd id: 88| 804.14|1476454985|
|2017-11-28 19:00:00|1734117444|  386167994| Best Buy ccd id: 36| 1013.1| 299170184|
|2017-12-19 19:00:00|1734118944| 1429095612|  Wal-Mart arc id: 5| 806.88| 553082198|
|2017-12-09 19:00:00|1734119639|  511877722|  Wendy's arc id: 84|1379.26| 668825839|
|2017-12-07 19:00:00|1734118539| 1996661856|  Wendy's ppd id: 

In [15]:
df_parquet = spark.read.format("parquet").load("data/parquet/sales_data.parquet")
df_parquet.show()

+-------------------+----------+-----------+--------------------+-------+----------+
|      transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+-------------------+----------+-----------+--------------------+-------+----------+
|2017-12-02 19:00:00|1734117000| 1953761884|   Target ppd id: 95| 422.92| 320106707|
|2017-11-25 19:00:00|1734117016|  644879053|       Home Depot 55| 659.82|1185242216|
|2017-12-16 19:00:00|1734117666|  902350112|         Walgreen 26| 1637.1| 573392624|
|2017-12-19 19:00:00|1734118338| 2077350195|           Costco 36|2277.63| 442865762|
|2017-11-28 19:00:00|1734117444|  386167994| Best Buy ccd id: 36| 1013.1| 299170184|
|2017-12-21 19:00:00|1734117885|  562903158|     unkn ppd id: 46| 797.24| 193309105|
|2017-12-23 19:00:00|1734118164|  644879053|           Costco 16|1658.36|1881132953|
|2017-12-16 19:00:00|1734117252| 2077350195|            Kings 25|1985.48|1760151621|
|2017-12-21 19:00:00|1734117416| 1429095612|DineEquity ppd id...|

In [16]:
# Benefits of Columnar Storage

# Lets create a simple Python decorator - {get_time} to get the execution timings
# If you dont know about Python decorators - check out : https://www.geeksforgeeks.org/decorators-in-python/
import time

def get_time(func):
    def inner_get_time() -> str:
        start_time = time.time()
        func()
        end_time = time.time()
        return (f"Execution time: {(end_time - start_time)*1000} ms")
    return inner_get_time()

In [17]:
# here we are using the decorator to get the execution time of reading a full parquet file.
# the working of decorator is become x = get_time(x) where get_time returns another function inner_get_time
# thet inner function inner_get_time calls holds the original function x() and calculates the execution time.
# but the inner_get_time returns a string so the x is now a string but all calculation happened in the inner_get_time function.
# so just calling print(x) will give us the execution time of reading the parquet file.
@get_time
def x():
    df_parquet = spark.read.format("parquet").load("data/parquet/sales_data.parquet")
    df_parquet.count()
print(x)

Execution time: 410.2346897125244 ms


In [ ]:
# selecting a single column from the parquet file and counting the number of rows in that column  
# used only lesser time than counting the number of rows in the entire parquet file.
# because the parquet file is a columnar storage format, it can read only the required column and count the number of rows in that column.
@get_time
def x():
    df_parquet = spark.read.format("parquet").load("data/parquet/sales_data.parquet")
    df_parquet.select("trx_id").count()
print(x)

Execution time: 381.49523735046387 ms


In [ ]:
# How to read a recursive parquet file in Spark

"""
data/parquet/sales_recursive/
└── sales_1/
    ├── 1.parquet
    └── sales_2/
        └── 2.parquet
"""

# reading one file, one level deep
df_1 = spark.read.format("parquet").load("data/parquet/sales_recursive/sales_1/1.parquet")
df_1.show()

+-------------------+----------+-----------+-----------------+------+---------+
|      transacted_at|    trx_id|retailer_id|      description|amount|  city_id|
+-------------------+----------+-----------+-----------------+------+---------+
|2017-12-02 19:00:00|1734117000| 1953761884|Target ppd id: 95|422.92|320106707|
+-------------------+----------+-----------+-----------------+------+---------+



In [22]:
# reading one file, two levels deep
df_1 = spark.read.format("parquet").load("data/parquet/sales_recursive/sales_1/sales_2/2.parquet")
df_1.show()

+-------------------+----------+-----------+-------------+------+----------+
|      transacted_at|    trx_id|retailer_id|  description|amount|   city_id|
+-------------------+----------+-----------+-------------+------+----------+
|2017-11-25 19:00:00|1734117016|  644879053|Home Depot 55|659.82|1185242216|
+-------------------+----------+-----------+-------------+------+----------+



In [23]:
# recursiveFileLookup walks the whole tree - both files come back as ONE DataFrame
df_1 = (spark.read.format("parquet")
        .option("recursiveFileLookup", True)
        .load("data/parquet/sales_recursive/"))
df_1.show()

+-------------------+----------+-----------+-----------------+------+----------+
|      transacted_at|    trx_id|retailer_id|      description|amount|   city_id|
+-------------------+----------+-----------+-----------------+------+----------+
|2017-12-02 19:00:00|1734117000| 1953761884|Target ppd id: 95|422.92| 320106707|
|2017-11-25 19:00:00|1734117016|  644879053|    Home Depot 55|659.82|1185242216|
+-------------------+----------+-----------+-----------------+------+----------+

